## T3:  Graphene
We now turn to graphene, which is a well-known hexagonal crystal structure. The new thing we will have to deal with for this example, is the need for $\vec{k}$-point sampling. It gives us some more equations to propagate, but it is just more of the same type of problem as we dealt with in the chain-example. Here we just sample five points and look at the differences. Beyond this, the procedure is the same. We proceed to do much of what has been covered previously in the next lines of code, and the geometry creation, Fermi-expansion and calls to TBtrans should be familiar. The calculation is very coarse and should not be taken as a good graphene calculation, but as a demonstration of how to propagate decoupled systems

In [ ]:
import sisl
import matplotlib.pyplot as plt
import numpy as np
from Zandpack.TimedependentTransport import TD_Transport as TDT
import matplotlib.image as img
plt.rcParams['figure.figsize']  = [12,8]
%matplotlib inline
tx     = 3
ty     = 2
t_elec = -2.7
t_dev  = -2.7

g = sisl.geom.graphene(orthogonal = True).tile(ty,1)
geom_dev = g.tile(tx+2,0)#.add_vacuum(10,1)
geom_em  = g.copy()
geom_ep  = g.move((tx+1) * g.cell[0,:])

# sisl.plot(geom_dev); plt.axis('equal'); plt.show()
eta = 1e-1
line = np.vstack([np.linspace(-5,5,75) + 1j * eta]*2)
Test = TDT([geom_em,geom_ep], geom_dev, kT_i = [0.05, 0.05])
Test.Make_Contour(line, 12, pole_mode = 'JieHu2011')

ky = 8
Test.Electrodes( semi_infs = ['-a1', '+a1'] , kp = [[50,ky,1],[50,ky,1]])
Test.make_device(elec_inds = [[i for i in range(geom_em.na)], 
                              [4*(tx+1)*ty+i for i in range(geom_em.na)]], 
                 k = [1,ky,1], k_tbtrans = [1,ky,1])

elec = sisl.Hamiltonian(geom_em)
elec.construct([[0.1, 1.5], 
                [0,   t_dev] ])


Test.run_electrodes(manual_H = [elec, elec])
dev_H = sisl.Hamiltonian(geom_dev)
dev_H.construct([[0.1, 1.5], 
                 [0,   t_dev] ])

Test.run_device(manual_H = dev_H)
Test.read_data()
plt.show()
plt.plot(Test.tbtT[Test.sampling_idx[0]], label = 'TBtrans Transmission')
plt.xlabel('Energy')
plt.legend()
plt.show()

In [ ]:
Test.Device.Visualise()
Test.Inspect_Transmission(0,1)
#Test.Lowdin[1].Block(1,1)

In [ ]:
dev_H.nsc

### Fitting
When fitting the peaked structure of the contact self-energies, many Lorentzians will end up in a relatively small energy-range. This is good to fit the peaks, but also introduces a high degree of linear dependence between the fitting functions. This results in very large eigenvalues (which we will see shortly) that still fits very well, but are computationally tricky to handle in the propagation. This can be understood as you can cancel Lorentzian with a large weight with an almost identical Lorentzian with the negative weight. Therefore a term that punishes overlap between the Lorentzian basis functions can be added through the "alpha_PO" keyword and introduces an Ad-Hoc term that drives the Lorentzian poles away from each other:

$
Error(\{\epsilon_i, \gamma_i\}_i) = \int  |\mathbf{f}(E) - \sum_{i}\mathbf{\Gamma}_iL(E, \epsilon_i, \gamma_i)|^2 \mathrm{d}E  + \mathrm{alpha\_PO}\cdot\mathrm{Overlap term}
$

In [ ]:
#Test.reset_all_fits() # Comment in to reset fits!
NL = 10
min_tol  = -0.0*np.ones(NL)[None,:][np.arange(30)*0,:]
min_tol1 =  min_tol.copy()
min_tol2 =  min_tol.copy()

def run_mini(its):
    Test.Fit(fact = 1.0, Fallback_W = 30.0, NumL = NL,
          fit_mode      = 'all',
          force_PSD     = True,
          force_PSD_tol = [min_tol1, min_tol2],
          use_analytical_jac = True,
          min_method = 'SLSQP',
          ebounds = (-10, 10),
          wbounds = (0.01, 0.8),
          gbounds = (None, None),
          tol = -1,
          options = {'disp':True,'maxiter':its, 
                     'gtol':1e-10,
                     'ftol':1e-10,
                     'iprint':1
                     },
          fit_real_part = False,
          specific_bounds = None,#[{(0 ,2) :[(0.1, 0.11), (4,5)]}, {(0 ,5) :[(-0.1, 0.1), (4,5)]}], 
          alpha_PO = 0.01, 
          )


In [ ]:
run_mini(500)

### The self-energies and transmission

We've proceeded a bit faster now than previously, but we are now at the point where we have the self-energy fits, and we can proceed with the propagation when we wish. We'll however just dwell a but longer with the self-energies. We now have a k-pont sampling we'll have to consider. The transmission is still inspected using the same function as has been used previously, but the Inspect_SE_lorentzian_fit" has the "ik" keyword, which is 0 by default and picks out that particular k-index. We sampled 10-k-points in TBtrans, but because of time-reversal symmetry, there are five district points, which you'll now examine. 

       1. Change ik from 0 to 5 below:
       2. Inspect the transmission function (It is very poorly fitting)

In [ ]:
#Test.Inspect_transmission_from_SE_fit(eta = 1e-2); plt.show()
#                              # first five previously explained,  k-idx , min,max energies      
print(Test.fitted_lorentzians[1].is_zero.shape)
IK = 1
Test.Inspect_Lorentzian_fit(1,11,11,0,0,  ik = IK, Emin = -7, Emax = 7,n_samples=1000)
plt.show()
Test.Inspect_Lorentzian_fit(1,11,11,1,0,  ik = IK, Emin = -7, Emax = 7,n_samples=1000)
plt.show()
Test.Inspect_Lorentzian_fit(0,0,0,1,1,    ik = IK, Emin = -7, Emax = 7,n_samples=1000)
plt.show()
Test.Inspect_Lorentzian_fit(0,0,0,1,0,    ik = IK, Emin = -7, Emax = 7,n_samples=1000)


In [ ]:
Test.Inspect_transmission_from_hilbert_transform(E=np.linspace(-5,5,100),eta=1e-1)

### Timepropagation
We now go ahead and propagate as we have done previously. We start out with the zero-bias and see how this turns out

In [ ]:
Test.tofile('Graphene')

In [ ]:
Test.Inspect_Poles(0, ik = 0); plt.ylim([0,3])

In [ ]:
!mpirun -np 3 zand Dir=$PWD

In [ ]:
!td_info Dir=$PWD file=Graphene plotcurrent=1 label=Current format=png ik=1 tmin=-20 tmax=100
im = img.imread('CurrentCurrentplot.png')
plt.rcParams['figure.figsize']=(15,9)
plt.imshow(im)


Why do the different K-points respond differently?


        - Consider the Gammas  we plotted before and the Hamiltonian


Go to your terminal and get familiar with the td_info tool.
        
        - try plotdevicecharge=1 
        - try plotdm=1 dm_inds=0,0
You can also use the td_info tool while the calculation is running in another terminal. Its useful for monitoring calculations

In [ ]:
!td_info --help

### SCF and psinought
The initial noise you see in some of the calculations, we can get rid of.
     
     1. Copy the Graphene folder and make a new one called "Gr" 
     2. Go to Inititial.py and set name='Gr' instead
     3. The next steps we do from jupyter, but you can do them in the terminal 
        just as well
Run the SCF code.  This determines the initial density matrix. 
Run the psinought code. This determines the initial $\psi_{k,\alpha x c}$ and $\Omega_{k,\alpha x c\alpha' x' c'}$.
Run SuperZand again, but with the Gr initial state instead. For this do:
    
    1. Go to Initial.py
    2. set usesave=True  
    3. set LoadFromFull=True
    4. Run SuperZand

In [ ]:
!SCF Dir=$PWD file=Gr

In [ ]:
!psinought Dir=$PWD file=Gr add_random=True

In [ ]:
!mpirun -np 3 SuperZand Dir=$PWD

In [ ]:
!td_info Dir=$PWD file=Gr plotcurrent=1 label=Current format=png ik=1
im = img.imread('CurrentCurrentplot.png')
plt.rcParams['figure.figsize']=(15,9)
plt.imshow(im)

### Learned Points

    - Multiple k-points (useful for propagating any quantum numbers that dont mix.  Also spin)
    - Fitting with the built-in fitting tool
    - Using the SCF tool can save time
    - Using the psinought tool can save time
    - Remembering to use "usesave=True" and "LoadFromFull=True" to utilize the output of psinought
    
    